# 01 EDA und Feature-Auswahl

Ziel: Dry-Bean-Datensatz laden, Klassenverteilung und Feature-Korrelationen untersuchen und eine erste Auswahl von 5 bis 10 Features begruenden.

## Forschungsfrage

Inwiefern lassen sich Dry-Bean-Sorten anhand weniger morphologischer Bildmerkmale zuverlaessig klassifizieren, und welche Merkmale erklaeren die Entscheidungen verschiedener Multiclass-Klassifikatoren am staerksten?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## Daten laden

Der Datensatz wird lokal aus `DryBeanDataset/Dry_Bean_Dataset.xlsx` geladen.

In [ ]:
data_path_candidates = [
    Path("DryBeanDataset/Dry_Bean_Dataset.xlsx"),
    Path("../DryBeanDataset/Dry_Bean_Dataset.xlsx"),
]
data_path = next(path for path in data_path_candidates if path.exists())

df = pd.read_excel(data_path)
X = df.drop(columns="Class")
y = df["Class"]

df.head()

In [ ]:
df.shape, df.isna().sum().sum(), df["Class"].value_counts()

## Klassenverteilung

Die Klassenverteilung ist wichtig, weil Accuracy bei unausgeglichenen Klassen alleine nicht ausreicht. Deshalb wird spaeter auch Macro-F1 verwendet.

In [ ]:
plt.figure(figsize=(9, 4))
sns.countplot(data=df, x="Class", order=df["Class"].value_counts().index)
plt.title("Klassenverteilung im Dry Bean Dataset")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## Korrelationen

Stark korrelierte Features liefern oft aehnliche Information. Fuer eine interpretierbare Feature-Auswahl sollten redundante Groessenmerkmale nicht alle gleichzeitig verwendet werden.

In [ ]:
corr = X.corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True)
plt.title("Korrelationsmatrix der Features")
plt.tight_layout()
plt.show()

In [ ]:
upper = corr.abs().where(np.triu(np.ones(corr.shape), k=1).astype(bool))
high_corr_pairs = (
    upper.stack()
    .reset_index()
    .rename(columns={"level_0": "feature_1", "level_1": "feature_2", 0: "abs_corr"})
    .query("abs_corr >= 0.90")
    .sort_values("abs_corr", ascending=False)
)
high_corr_pairs

## Pruefung redundanter Formmerkmale

Vor der Feature-Auswahl wird geprueft, ob einzelne Formmerkmale nahezu dieselbe Information enthalten. Besonders relevant ist `ShapeFactor3`, da es im Datensatz praktisch `Compactness` im Quadrat entspricht.

In [ ]:
max_abs_diff = (df["ShapeFactor3"] - df["Compactness"] ** 2).abs().max()
max_abs_diff

## Erste Feature-Auswahl

Vorschlag fuer eine erste, interpretierbare Auswahl:

- `Area`: Groesse der Bohne
- `Perimeter`: Umfang der Bohne
- `AspectRation`: Verhaeltnis von Laenge zu Breite
- `Compactness`: Kompaktheit der Form
- `roundness`: Rundheitsmerkmal
- `ShapeFactor1`, `ShapeFactor2`, `ShapeFactor4`: ergaenzende Formfaktoren

Diese Auswahl kombiniert Groesse und Form, vermeidet aber mehrere stark redundante Groessenfeatures wie `Area`, `ConvexArea` und `EquivDiameter` gleichzeitig. `ShapeFactor3` wird nicht aufgenommen, da es im Datensatz praktisch `Compactness` im Quadrat entspricht.

In [ ]:
selected_features = [
    "Area",
    "Perimeter",
    "AspectRation",
    "Compactness",
    "roundness",
    "ShapeFactor1",
    "ShapeFactor2",
    "ShapeFactor4",
]

df[selected_features + ["Class"]].head()

## Erste Modellversuche zur Feature-Auswahl

Fuer die Abgabe am 05.05. werden mehrere Feature-Sets mit den drei vorausgewaehlten Modellen getestet. Ziel ist nicht die finale Optimierung, sondern eine erste empirische Begruendung der Feature-Auswahl.

In [ ]:
feature_sets = {
    "all_16_features": list(X.columns),
    "domain_8_features": selected_features,
    "compact_6_features": [
        "Area", "AspectRation", "Eccentricity", "roundness",
        "ShapeFactor1", "ShapeFactor2",
    ],
    "mixed_10_features": [
        "Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
        "AspectRation", "Eccentricity", "roundness", "Compactness",
        "ShapeFactor1", "ShapeFactor2",
    ],
}

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, random_state=RANDOM_STATE, class_weight="balanced", n_jobs=-1
    ),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

rows = []
for feature_set_name, features in feature_sets.items():
    X_subset = X[features]
    X_train, X_test, y_train, y_test = train_test_split(
        X_subset, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    for model_name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        rows.append({
            "feature_set": feature_set_name,
            "n_features": len(features),
            "model": model_name,
            "accuracy": accuracy_score(y_test, y_pred),
            "macro_f1": f1_score(y_test, y_pred, average="macro"),
        })

feature_experiment_results = (
    pd.DataFrame(rows)
    .sort_values(["feature_set", "macro_f1"], ascending=[True, False])
)
feature_experiment_results

In [ ]:
feature_experiment_summary = (
    feature_experiment_results
    .groupby(["feature_set", "n_features"])[["accuracy", "macro_f1"]]
    .mean()
    .sort_values("macro_f1", ascending=False)
)
feature_experiment_summary

## Erste Empfehlung

Alle 16 Features erzielen erwartungsgemaess die beste reine Modellleistung. Fuer die wissenschaftliche Arbeit ist aber eine reduzierte und fachlich interpretierbare Auswahl sinnvoller. Als Startpunkt wird deshalb `domain_8_features` verwendet, weil dieses Set in den ersten Versuchen besser als die 10-Feature-Variante abschneidet und gleichzeitig kompakter ist. Es kombiniert Groesse, Umfang, Laenglichkeit, Rundheit, Kompaktheit und Formfaktoren. Stark zusammenhaengende Groessen- und Achsenmerkmale wie `ConvexArea`, `EquivDiameter`, `MajorAxisLength` und `MinorAxisLength` werden bewusst nicht aufgenommen. `ShapeFactor3` wird ebenfalls nicht verwendet, weil es praktisch dieselbe Information wie `Compactness` in quadrierter Form enthaelt.